# 02 - Exploratory data analysis (retail churn)

Which customers churn, when, and which differences are statistically real.
Everything is computed with pandas from `config.FEATURES_CSV`, so no SQL views or pre-rendered figures are needed.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from src import configs   # save configs.py as src/config.py

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

T = configs.TARGET
feats = pd.read_csv(configs.FEATURES_CSV)
print(feats.shape, f"| overall churn rate: {feats[T].mean():.1%}")
feats[configs.MODEL_FEATURES + [T]].isna().sum().loc[lambda s: s > 0]

(1000, 25) | overall churn rate: 52.6%


KeyError: "['support_contact_rate', 'rescency_ratio', 'spend_to_income_ratio'] not in index"

## Churn by segment

In [ ]:
def bucket(col, q=5):
    """Use the column as-is if it has few distinct values, else cut into quantile bins."""
    s = feats[col]
    return s if s.nunique() <= 10 else pd.qcut(s, q, duplicates="drop")

def churn_by(col, q=5):
    out = feats.groupby(bucket(col, q), observed=True)[T].agg(customers="size", churn_rate="mean")
    out["churn_rate"] = (out["churn_rate"] * 100).round(1)
    return out

seg_cols = configs.CATEGORICAL_FEATURES + ["satisfaction_score", "num_of_support_contacts", "age"]
for c in seg_cols:
    print(f"--- {configs.FEATURE_LABELS[c]}"); print(churn_by(c).to_string()); print()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, c in zip(axes.ravel(), seg_cols):
    d = churn_by(c)
    ax.bar(d.index.astype(str), d["churn_rate"])
    ax.axhline(feats[T].mean() * 100, color="k", ls="--", lw=1)
    ax.set_title(configs.FEATURE_LABELS[c]); ax.set_ylabel("churn %")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig(configs.FIGURES_DIR / "eda_churn_by_segment.png", dpi=120)
plt.show()

## Do tenure and recency affect churn? When do customers leave?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, c in zip(axes, ["years_as_customer", "last_purchase_days_ago"]):
    d = churn_by(c, q=10)
    ax.plot(range(len(d)), d["churn_rate"], marker="o")
    ax.set_xticks(range(len(d))); ax.set_xticklabels(d.index.astype(str), rotation=45, ha="right")
    ax.set_title(f"Churn rate by {configs.FEATURE_LABELS[c]} (deciles)"); ax.set_ylabel("churn %")
plt.tight_layout()
plt.savefig(configs.FIGURES_DIR / "eda_tenure_recency_curve.png", dpi=120)
plt.show()

## Do spend and purchase behaviour affect churn?

In [ ]:
cols = ["total_spend", "spend_per_year", "num_of_purchases", "average_transaction_amount",
        "num_of_returns", "annual_income"]
feats.groupby(T)[cols].median().round(1)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, c in zip(axes.ravel(), cols):
    for k, g in feats.groupby(T):
        ax.hist(g[c].clip(upper=g[c].quantile(0.99)), bins=30, alpha=0.5, density=True, label=f"churn={k}")
    ax.set_title(configs.FEATURE_LABELS[c]); ax.legend()
plt.tight_layout()
plt.savefig(configs.FIGURES_DIR / "eda_distributions.png", dpi=120)
plt.show()

## Are the differences real? Significance tests

In [ ]:
def cramers_v(table):
    chi2 = stats.chi2_contingency(table)[0]
    n = table.to_numpy().sum()
    return np.sqrt(chi2 / (n * (min(table.shape) - 1)))

rows = []
# categorical (plus satisfaction_score, which is a small ordinal scale)
for col in configs.CATEGORICAL_FEATURES + ["satisfaction_score"]:
    table = pd.crosstab(feats[col], feats[T])
    chi2, p, dof, _ = stats.chi2_contingency(table)
    rows.append({"feature": col, "test": "chi-square", "p_value": p,
                 "effect_size": cramers_v(table), "effect": "Cramer's V"})
# numeric
for col in [c for c in configs.NUMERIC_FEATURES if c != "satisfaction_score"]:
    a, b = feats.loc[feats[T] == 1, col].dropna(), feats.loc[feats[T] == 0, col].dropna()
    u, p = stats.mannwhitneyu(a, b)
    rows.append({"feature": col, "test": "Mann-Whitney U", "p_value": p,
                 "effect_size": abs(1 - 2 * u / (len(a) * len(b))), "effect": "rank-biserial r"})

tests = pd.DataFrame(rows).sort_values("effect_size", ascending=False)
tests["significant (p<0.001)"] = tests["p_value"] < 0.001
tests.round(4)

*Write your reading here after running: which features clear p < 0.001, which don't, and
how the effect sizes rank the drivers. Check that `gender` is not a meaningful driver.*

## Correlation between numeric features

In [ ]:
corr = feats[configs.NUMERIC_FEATURES + [T]].corr()
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
fig.colorbar(im); plt.tight_layout()
plt.savefig(configs.FIGURES_DIR / "eda_correlation.png", dpi=120)
plt.show()

pairs = (corr.where(np.triu(np.ones(corr.shape), 1).astype(bool)).stack()
             .loc[lambda s: s.abs() > 0.7].sort_values(key=abs, ascending=False))
print("Highly correlated pairs (|r| > 0.7):"); print(pairs.round(2).to_string())
print("\nCorrelation with churn:"); print(corr[T].drop(T).sort_values(key=abs, ascending=False).round(3).to_string())

Highly correlated pairs (e.g. `total_spend` vs `spend_per_year`, `num_of_purchases` vs
`purchases_per_year`) are fine for trees and regularised models, but importance is *shared* between
them, so SHAP explanations are better summed per business feature than read column by column.

## Key findings

*Fill in from the outputs above (numbers, not impressions):*

1. **Recency:** churn rate in the most-dormant decile vs the most recent decile.
2. **Tenure:** are newer or older customers more likely to leave?
3. **Satisfaction / support contacts:** churn at low vs high satisfaction; effect of repeated support contacts.
4. **Returns and spend:** do high-return or low-spend customers churn more?
5. **Promotion response / email opt-in:** does engagement with marketing separate churners?
6. **Riskiest cell:** the combination of two features with the highest churn rate.